In [ ]:
!pip install datasets
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is 

In [ ]:
import torch
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer
import re

In [ ]:
def string_preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def get_edit_distance(org_sentence, pred_sentence):
    org_words, pred_words = org_sentence.split(), pred_sentence.split()
    len_org, len_pred = len(org_words), len(pred_words)
    dp = dict()

    for index in range(len_pred + 1):
        dp[(index, 0)] = index
    for index in range(len_org + 1):
        dp[(0, index)] = index

    for index1 in range(1, len_pred + 1):
        for index2 in range(1, len_org + 1):
            if pred_words[index1 - 1] == org_words[index2 - 1]:
                dp[(index1, index2)] = dp[(index1 - 1, index2 - 1)]
            else:
                dp[(index1, index2)] = 1 + min(
                    dp[(index1 - 1, index2)],
                    dp[(index1, index2 - 1)],
                    dp[(index1 - 1, index2 - 1)]
                )

    return dp[(len_pred, len_org)]


def compute_word_error_rate(original_texts, predicted_texts):
    total_words = 0
    total_edit_distances = 0
    for index in range(len(original_texts)):
        org_sentence = string_preprocess(original_texts[index])
        pred_sentence = string_preprocess(predicted_texts[index])
        edit_distance = get_edit_distance(org_sentence, pred_sentence)
        total_edit_distances += edit_distance
        total_words += len(org_sentence.split())

    return float(total_edit_distances) / total_words

In [ ]:
MODEL_NAME    = "openai/whisper-base"
SPLIT         = "validation"
TARGET_SR     = 16_000
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DEVICE

device(type='cuda')

In [ ]:
ds_small = (
        load_dataset("edinburghcstr/edacc", split=SPLIT)
        .cast_column("audio", Audio(sampling_rate=TARGET_SR))
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

(…)-00000-of-00006-17ec18f4d1c3d587.parquet:   0%|          | 0.00/457M [00:00<?, ?B/s]

(…)-00001-of-00006-6f3978e4f6163671.parquet:   0%|          | 0.00/478M [00:00<?, ?B/s]

(…)-00002-of-00006-5a1d86079a8c9228.parquet:   0%|          | 0.00/446M [00:00<?, ?B/s]

...

(…)-00007-of-00010-ea5ed4464ecff3c9.parquet:   0%|          | 0.00/439M [00:00<?, ?B/s]

(…)-00008-of-00010-d1aa19b51ad423da.parquet:   0%|          | 0.00/409M [00:00<?, ?B/s]

(…)-00009-of-00010-648ce5002d124496.parquet:   0%|          | 0.00/299M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/9848 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9289 [00:00<?, ? examples/s]

In [ ]:
ds_small

Dataset({
    features: ['speaker', 'text', 'accent', 'raw_accent', 'gender', 'l1', 'audio'],
    num_rows: 9848
})

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model     = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE).eval()

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 512)
      (layers): ModuleList(
        (0-5): 6 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=False)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          

In [ ]:
import time

references, hypotheses = [], []
index = 0
print(f"start, {time.time()}")
for example in ds_small:
    wav = example["audio"]["array"]
    references.append(example["text"])

    inputs = processor(wav, sampling_rate=TARGET_SR, return_tensors="pt")
    input_feats = inputs.input_features.to(DEVICE)

    pred_ids = model.generate(input_feats)

    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    hypotheses.append(pred_text)

    index += 1
    if index % 100 == 0:
        print(f"{index}, {time.time()}")

overall_wer = compute_word_error_rate(references, hypotheses) * 100
print(f"\nOverall WER on {len(ds_small)} examples: {overall_wer:.2f}%")

start, 1746043433.8670793
100, 1746043461.0392783
200, 1746043480.7249608
300, 1746043500.8686876
400, 1746043518.6461554
500, 1746043540.264063
600, 1746043558.8811233
700, 1746043579.7653725
800, 1746043601.8536344
900, 1746043620.223686
1000, 1746043640.576339
1100, 1746043665.0481746
1200, 1746043691.0434184
1300, 1746043710.7447278
1400, 1746043732.58825
1500, 1746043752.0537333
1600, 1746043777.6038938
1700, 1746043793.1258368
1800, 1746043809.1085768
1900, 1746043825.0357754
2000, 1746043844.5965958
2100, 1746043866.4974613
2200, 1746043897.6601605
2300, 1746043931.694006
2400, 1746043957.2822309
2500, 1746043973.627007
2600, 1746043988.5188444
2700, 1746044003.1639833
2800, 1746044019.301009
2900, 1746044043.8585334
3000, 1746044079.3113081
3100, 1746044107.4004874
3200, 1746044132.453469
3300, 1746044162.2450397
3400, 1746044185.53948
3500, 1746044204.235376
3600, 1746044221.347482
3700, 1746044239.7458217
3800, 1746044257.543139
3900, 1746044278.85691
4000, 1746044306.5851111

In [ ]:
result = []

for ref, hyp in zip(references, hypotheses):
    result.append([ref, hyp])

print(result[:5])

[['C ELEVEN DASH P ONE', ' C11-P1.'], ['C ELEVEN DASH P TWO', ' c11-p2.'], ['IGNORE_TIME_SEGMENT_IN_SCORING', ' Please call Stella, ask her to bring these things with her from the store. Six spoons of fresh snow peas, five thick slabs of blue cheese, and maybe a snack for her brother Bob. We also need a small plastic snake and a big toy frog for the kids. She can scoop these things into three red bags and we will go meet her Wednesday at the train station.'], ['IGNORE_TIME_SEGMENT_IN_SCORING', ' Please call Stella, ask her to bring these things with her from the store. Six spoons of fresh snow peas, five thick slabs of blue cheese, and maybe a snack for her brother Bob. We also need a small plastic snake and a big toy frog for the kids. She can scoop these things into three red bags, and we will go meet her Wednesday at the train station.'], ['OKAY NOW FOR A REGULAR CONVERSATION SO UH WOULD YOU RATHER GO TO THE BEACH TODAY OR DO CLUB GOLF DISCUSS', ' Okay, now for a regular conversatio

In [ ]:
import json
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
output_path = '/content/drive/My Drive/result.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=4)
